# 🚁 Notebook 03 — Open Loop vs Closed Loop

### Giving the drone *eyes* so it can correct itself

At the end of Notebook 02 you fought the slider trying to hover a drone with a fixed thrust — and
lost. The moment mass, gravity, or a gust changed, the drone drifted and **nothing corrected it**.
That approach has a name: **open-loop control** (command something and hope).

This notebook introduces the single most important idea in all of control: **feedback**. We give
the drone a **sensor** to measure where it actually is, compare that to where we *want* it, and use
the difference to decide what to do. That's a **closed loop** — and it changes everything.

---

## 🎯 Learning objectives
- Clearly distinguish **open-loop** vs **closed-loop (feedback)** control.
- Understand a **sensor**, a **target** (setpoint), the **current** value, and the **error**.
- Explain *why* open-loop hover fails and feedback fixes it.
- Build a first, crude **on/off (bang-bang)** feedback controller and see it keep the drone near
  target — while wobbling.
- Feel exactly *why* we'll want something smoother (the proportional controller, Notebook 04).

## 🧩 Prerequisites
- Notebooks 01–02 (simulation, Euler, forces, gravity, thrust). Quick recap below.

## ⏱️ Estimated time
About **40–55 minutes**.

## 🧠 Concepts you know so far
Position · Velocity · Acceleration · Euler integration · Force · Mass · Newton's law · Gravity · Thrust · Net force · Manual (open-loop) hover

## 🔜 Concepts you'll learn here
Open loop · Closed loop · Feedback · Sensor · Target / setpoint · Error · Bang-bang control


### 🔁 Quick recap (Notebooks 01–02)

Our drone simulator, in one breath:

```
net_force = thrust - mass * g          # forces add up (up +, down -)
a = net_force / mass                   # Newton's second law
v = v + a * dt                         # Euler: acceleration -> velocity
x = x + v * dt                         # Euler: velocity     -> position
```

The problem: a **fixed** `thrust` can't keep the drone hovering. Now we'll make `thrust` **change
based on what the drone measures.** Run setup and let's build it.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
# We only use numpy, matplotlib and ipywidgets in this whole course.
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

# Make animations play as an embedded video/player (works in Google Colab).
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80   # MB, so longer animations don't get cut off
plt.rcParams["figure.figsize"] = (8, 4)      # a comfortable default plot size

# ipywidgets gives us interactive sliders.
from ipywidgets import interact, FloatSlider, IntSlider

# This line helps sliders/widgets show up correctly in Google Colab.
# It does nothing (harmlessly) if you are NOT in Colab.
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready to go!")


## 1. Open loop = "command and hope"

**Open-loop control** means you decide the command **ahead of time** and never look at the result.

Everyday examples:
- 🍞 A basic toaster runs for a set 2 minutes — it doesn't *look* at how brown the toast is.
- 🎯 Throwing a dart with your eyes closed — you commit to a throw and hope.
- 🚁 Our Notebook-02 drone with a fixed thrust — it never checks its own altitude.

Open loop works *only* if you know the world perfectly and nothing ever disturbs it. The real world
is never like that. Let's show the failure crisply: pick the "perfect" hover thrust, then imagine
the drone actually weighs **5% more** than we thought (a common real mismatch), and watch it sink.


In [ ]:
def simulate_open_loop(commanded_thrust, true_mass, g=9.8,
                       start_height=10.0, dt=0.02, total_time=10.0):
    """Fixed thrust, no feedback. The controller THINKS the mass is 1.0 but the TRUE mass differs."""
    x, v, t = start_height, 0.0, 0.0
    times, xs = [], []
    for _ in range(int(total_time / dt)):
        times.append(t); xs.append(x)
        net_force = commanded_thrust - true_mass * g
        a = net_force / true_mass
        v = v + a * dt
        x = x + v * dt
        if x < 0: x, v = 0.0, 0.0
        t = t + dt
    return times, xs

# We compute the "perfect" thrust assuming mass = 1.0 kg:
assumed_mass = 1.0
commanded_thrust = assumed_mass * 9.8       # 9.8 N -- perfect IF the drone is really 1.0 kg

plt.figure(figsize=(8, 4.5))
for true_mass, colour in [(1.00, "seagreen"), (1.05, "crimson"), (0.95, "royalblue")]:
    tL, xL = simulate_open_loop(commanded_thrust, true_mass)
    plt.plot(tL, xL, color=colour, label=f"true mass = {true_mass} kg")
plt.axhline(10, color="gray", linestyle="--", label="target 10 m")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Open loop: a tiny 5% mass error and the drone drifts away")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

print("Only the perfectly-guessed mass (green) holds. 5% heavier -> sinks. 5% lighter -> climbs.")
print("The controller never SEES this, so it never corrects. That's the fatal flaw of open loop.")


## 2. Closing the loop: sensor, target, current, error

To fix this, the drone needs to **look at itself** and react. Four words unlock everything:

- **Sensor** — a device that *measures* the current altitude (e.g. a barometer or rangefinder).
  We'll call its reading `measured_altitude`.
- **Target** (also called **setpoint**) — where we *want* the drone to be, e.g. `10` m. Call it
  `target`.
- **Current** — where it *actually* is right now, `x`.
- **Error** — the heart of control. The gap between where we want to be and where we are:

$$ \text{error} = \text{target} - \text{current} $$

Read the sign carefully:
- error **> 0** → we're **below** the target → we need to go **up** (add thrust).
- error **< 0** → we're **above** the target → we need to go **down** (reduce thrust).
- error **= 0** → we're exactly on target → don't change anything.

**Everything** — P, I, and D — is just different clever ways of turning this one `error` number
into a thrust command. Get comfortable with `error = target − current` and the rest of the course
falls into place.


In [ ]:
# The error is trivial to compute -- but it is the beating heart of all control.
def compute_error(target, current):
    """How far off are we? Positive = below target (need to climb)."""
    return target - current

target = 10.0
for current in [4.0, 10.0, 13.0]:
    e = compute_error(target, current)
    if   e > 0: meaning = "below target -> should climb"
    elif e < 0: meaning = "above target -> should descend"
    else:       meaning = "exactly on target -> hold"
    print(f"current = {current:5.1f} m   error = {e:+5.1f}   ->  {meaning}")


## 3. The feedback loop, as a picture 🔁

Closed-loop control is a cycle the drone repeats **every single time step**:

```
        ┌──────────────────────────────────────────────┐
        │                                              │
        ▼                                              │
   ┌─────────┐   error    ┌────────────┐  thrust   ┌────────┐
   │ compare │──────────▶ │ controller │─────────▶ │ drone  │
   │ target  │            │ (decides   │           │ physics│
   │ vs      │            │  thrust)   │           │ moves  │
   │ current │            └────────────┘           └────────┘
   └─────────┘                                          │
        ▲                                               │
        │              sensor measures new altitude     │
        └───────────────────────────────────────────────┘
```

Round and round: **measure → compare (get error) → decide thrust → drone moves → measure again.**
Because it *keeps checking*, it can catch drift, fight wind, and cope with a wrong mass guess. The
only piece we haven't defined is the **controller** box — "given the error, what thrust?" Let's
build the simplest possible one.


## 4. The simplest feedback controller: **bang-bang** (on/off)

The crudest possible rule for the controller box:

> If we're **below** target → blast **maximum** thrust (go up!).
> If we're **above** target → cut thrust to **minimum** (fall down).

It's called **bang-bang** control (or on/off control) because it slams between two extremes — just
like a home thermostat clicking the heater fully ON or fully OFF. Let's code it and see it actually
*work*: unlike open loop, it now keeps the drone near the target even though we deliberately give it
a **wrong mass guess and a gust of wind**.


In [ ]:
def simulate_bang_bang(target=10.0, true_mass=1.2, g=9.8, wind=-1.0,
                       max_thrust=25.0, min_thrust=0.0,
                       start_height=0.0, dt=0.02, total_time=12.0):
    """On/off feedback controller. Note: it does NOT know the true mass or the wind --
    it only reacts to the measured error. That's the power of feedback."""
    x, v, t = start_height, 0.0, 0.0
    times, xs, thrusts = [], [], []
    for _ in range(int(total_time / dt)):
        # ---- SENSE ----
        measured_altitude = x                       # (perfect sensor for now; noise comes later)
        # ---- COMPARE ----
        error = target - measured_altitude
        # ---- DECIDE (bang-bang) ----
        if error > 0:                               # below target
            thrust = max_thrust                     # full blast up
        else:                                       # at or above target
            thrust = min_thrust                     # cut the motors

        times.append(t); xs.append(x); thrusts.append(thrust)
        # ---- ACT (drone physics) ----
        net_force = thrust - true_mass * g + wind   # wind is an unknown disturbance
        a = net_force / true_mass
        v = v + a * dt
        x = x + v * dt
        if x < 0: x, v = 0.0, 0.0
        t = t + dt
    return times, xs, thrusts

tL, xL, thrL = simulate_bang_bang()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(tL, xL, color="mediumvioletred"); ax1.axhline(10, color="seagreen", ls="--", label="target")
ax1.set_ylabel("altitude (m)"); ax1.set_title("Bang-bang feedback keeps it NEAR target (but wobbles)")
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(tL, thrL, color="darkorange"); ax2.set_ylabel("thrust (N)"); ax2.set_xlabel("time (s)")
ax2.set_title("...by slamming thrust fully ON / fully OFF"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Feedback WORKS: despite a wrong mass (1.2 kg) and wind, the drone stays around 10 m.")
print("But look at the altitude -- it never settles, it buzzes around the target forever.")


## 5. 🎬 Watch feedback fight to hold altitude

The drone below doesn't know its true mass or the wind — it only reacts to error. Watch it hunt
around the green target line, overshooting and dipping, thrust flickering on and off.


In [ ]:
tL, xL, thrL = simulate_bang_bang(total_time=12.0)
frame_ids = np.linspace(0, len(tL) - 1, 130).astype(int)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1); ax.set_ylim(0, 16); ax.set_xticks([])
ax.set_ylabel("altitude (m)"); ax.set_title("Bang-bang feedback")
ax.axhline(0, color="saddlebrown", lw=3)
ax.axhline(10, color="seagreen", ls="--", lw=2)              # target
drone, = ax.plot([], [], "o", color="mediumvioletred", markersize=26)
label = ax.text(-0.9, 14.5, "", fontsize=11)

def init():
    drone.set_data([], []); label.set_text(""); return drone, label

def update(f):
    i = frame_ids[f]
    drone.set_data([0], [xL[i]])
    label.set_text(f"t={tL[i]:.1f}s\nx={xL[i]:.2f} m\nthrust={thrL[i]:.0f} N")
    return drone, label

ani = animation.FuncAnimation(fig, update, frames=len(frame_ids), init_func=init,
                              blit=True, interval=45)
plt.close(fig)
HTML(ani.to_jshtml())


## 6. 🎛️ Feel it: feedback beats disturbances that open loop can't

Change the **true mass** and **wind** sliders. Notice the crucial win: with bang-bang **feedback**,
the drone stays near target for *any* of these values — because it reacts to what it measures. The
old open-loop drone would have drifted away for every one of these. Feedback is *robust* where open
loop is brittle.

The catch you'll also feel: the ride is **jittery**. It never smoothly settles — it keeps buzzing.
That's the weakness we'll cure next.


In [ ]:
def explore_bang(true_mass=1.2, wind=-1.0, target=10.0):
    tL, xL, _ = simulate_bang_bang(target=target, true_mass=true_mass, wind=wind, total_time=12.0)
    plt.figure(figsize=(8, 4.5))
    plt.plot(tL, xL, color="mediumvioletred", label="drone altitude")
    plt.axhline(target, color="seagreen", ls="--", label="target")
    plt.ylim(0, target * 1.7); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
    plt.title(f"true mass={true_mass} kg, wind={wind} N -> feedback still holds ~target")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

interact(explore_bang,
         true_mass=FloatSlider(min=0.6, max=2.0, step=0.1, value=1.2),
         wind=FloatSlider(min=-4.0, max=4.0, step=0.5, value=-1.0),
         target=FloatSlider(min=4.0, max=14.0, step=1.0, value=10.0));


## ✅ Summary — what you now understand

- **Open-loop** control commands blindly and can't correct for errors, wrong assumptions, or
  disturbances. It's brittle.
- **Closed-loop (feedback)** control **measures** the result, computes **error = target − current**,
  and adjusts the command. It's robust.
- The feedback cycle repeats every time step: **sense → compare → decide → act → sense again.**
- **Bang-bang** control is the crudest feedback rule (full ON below target, full OFF above). It
  *works* — the drone stays near target despite unknown mass and wind — but it's **jittery** and
  never settles smoothly.


## ⚠️ Common mistakes
- **Getting the error sign backwards.** `error = target − current`. If you flip it, your controller
  pushes the wrong way and the drone runs away. This is the #1 beginner bug.
- **Believing open loop "sometimes works."** It only works when your model is perfect and nothing
  disturbs the system — i.e. never, in reality.
- **Expecting bang-bang to be smooth.** By design it slams between extremes, so wobble is
  unavoidable. Smoothness needs a *proportional* response — next notebook.

## 🧭 Mental model
> *"Feedback = don't command blindly; keep measuring the gap between where you want to be and where
> you are, and keep nudging to close that gap."*

## 🌍 Real-world applications
Thermostats, cruise control, oven temperature, ship autopilots, insulin pumps, and every real
flight controller. All of them close the loop: measure, compare to a target, correct — forever.


## 🧪 Exercises

**Level 1 — Observe.** In `simulate_bang_bang`, set `wind = 3.0` (a strong updraft) and re-run the
plot. Does the drone still stay near 10 m?

<details><summary>▸ Hint</summary>
Feedback reacts to error regardless of the disturbance's source.
</details>
<details><summary>▸ Solution</summary>
Yes — it still hovers around 10 m, just with a slightly different wobble pattern. The controller
never "knows" about the wind; it simply keeps correcting the measured error. That robustness is the
entire point of feedback.
</details>

---
**Level 2 — Modify.** Change the bang-bang rule so that when above target it uses a small "gentle"
thrust of `8.0` N instead of `0.0`. Re-run. Does the wobble get smaller?

<details><summary>▸ Hint</summary>
Replace <code>min_thrust=0.0</code>'s effect: in the else-branch set <code>thrust = 8.0</code>.
A less extreme "off" state means less violent dropping.
</details>
<details><summary>▸ Solution</summary>
The wobble shrinks. Because the "off" state no longer cuts thrust to zero, the drone doesn't
plummet as hard when above target, so the oscillations are gentler. This is a hint that a
<b>smoothly varying</b> thrust (proportional to error) would be even better — exactly Notebook 04.
</details>

---
**Level 3 — Predict, then check.** In open loop with `commanded_thrust = 9.8` N and a true mass of
**0.9 kg**, will the drone rise or sink? Predict, then run `simulate_open_loop(9.8, 0.9)` and look.

<details><summary>▸ Hint</summary>
Hover needs <code>thrust = m*g = 0.9 × 9.8 = 8.82 N</code>. You're commanding 9.8 N.
</details>
<details><summary>▸ Solution</summary>
It **rises**: 9.8 N is more than the 8.82 N needed to hover a 0.9 kg drone, so there's leftover
upward net force and it climbs away. Open loop can't see or fix this.
</details>


## ❓ Quick quiz

**Q1.** A toaster on a fixed timer is an example of…
<details><summary>▸ Answer</summary>
**Open-loop** control — it never measures how brown the toast is.
</details>

**Q2.** `error = target − current`. The drone is at 12 m, target is 10 m. The error is…
<details><summary>▸ Answer</summary>
**−2** (negative → we're *above* target → we should descend).
</details>

**Q3.** Why does feedback handle an unknown wind that open loop can't?
<details><summary>▸ Answer</summary>
Feedback measures the *actual* altitude and corrects whatever error it sees — no matter the cause.
Open loop never looks, so it never notices the wind pushed it off target.
</details>

**Q4.** The main downside of bang-bang control is…
<details><summary>▸ Answer</summary>
It's **jittery / never settles** — it slams between full-on and full-off, so the drone perpetually
wobbles around the target instead of resting smoothly on it.
</details>


## 🔭 Preview of Notebook 04 — *Building the P Controller*

Bang-bang works but wobbles because it only knows two responses: *full blast* or *nothing*. What if
the thrust correction were **proportional** to the error — a big push when we're far below target,
a gentle nudge when we're close, and nothing when we're spot on? That single idea is the
**Proportional (P) controller**, the "P" in "PID". In Notebook 04 you'll build it one line at a
time, tune its gain `Kp`, and meet the classic behaviours every control engineer knows by heart:
**rise time, overshoot, oscillation, and settling.** 🚁

**Excellent — you now understand feedback, the core idea of the whole course!**
